In [4]:
import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import pprint

from IPython.display import display, HTML, Markdown
from pathlib import Path
from matplotlib.patches import Polygon

def add_module_path(module_dir_name="10_modules"):
	"""
	Search current working directory and its parents for a folder named `module_dir_name`
	and add it to sys.path (at front) so local modules can be imported reliably from notebooks.
	Returns the path that was added or None if not found.
	"""
	cwd = Path.cwd()
	# check a few likely candidates (cwd, cwd.parent, and all ancestors)
	candidates = [cwd / module_dir_name, cwd.parent / module_dir_name] + [p / module_dir_name for p in cwd.parents]
	seen = set()
	for cand in candidates:
		try:
			cand_resolved = cand.resolve()
		except Exception:
			continue
		if cand_resolved in seen:
			continue
		seen.add(cand_resolved)
		if cand.exists() and cand.is_dir():
			pstr = str(cand_resolved)
			if pstr not in sys.path:
				sys.path.insert(0, pstr)
			return pstr
	return None

# enable autoreload before importing local modules so changes are picked up automatically
%load_ext autoreload
%autoreload 2

added = add_module_path("10_modules")
if added:
	print("Added to sys.path:", added)
else:
	print("Warning: '10_modules' not found in current working directory or any parent. Current cwd:", Path.cwd())

# import the local helper module; any import error will be shown clearly
try:
	import dh_prep as dhp
except Exception as e:
	print("Failed to import 'dh_prep'. Ensure '10_modules/dh_prep.py' exists and is accessible. Error:", e)
	raise



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Added to sys.path: /home/fanie/03_develop/2026-03-05_geostats_ag/10_modules


In [8]:
if __name__ == "__main__":
    # df1: Assays (Notice the gap from 2.0 to 3.0!)
    df1 = pd.DataFrame({
        'HOLEID': ['DH001', 'DH001'],
        'FROM': [0.0, 3.0],
        'TO': [2.0, 5.0],
        'Au_ppm': [1.5, 4.2]
    })

    # df2: Lithology (Notice it starts at 1.0, leaving 0.0 to 1.0 unlogged!)
    df2 = pd.DataFrame({
        'HOLEID': ['DH001', 'DH001'],
        'FROM': [1.0, 4.0],
        'TO': [4.0, 6.0],
        'LITH': ['Oxide', 'Fresh']
    })

    # Run the subroutine
    merged_db = dhp.merge_intervals(df1, df2)
    print(merged_db.to_string())

  HOLEID  FROM   TO  LENGTH  Au_ppm   LITH
0  DH001   1.0  2.0     1.0     1.5  Oxide
1  DH001   3.0  4.0     1.0     4.2  Oxide
2  DH001   4.0  5.0     1.0     4.2  Fresh


In [16]:
dh1 = dhp.fill_drillhole_gaps(df1)
dh2 = dhp.fill_drillhole_gaps(df2)

dh1,dh2 = dhp.align_end_of_hole(dh1, dh2)

merged_db = dhp.merge_intervals(dh1, dh2)
print(merged_db.to_string())


  HOLEID  FROM   TO  LENGTH  Au_ppm   LITH
0  DH001   0.0  1.0     1.0     1.5    NaN
1  DH001   1.0  2.0     1.0     1.5  Oxide
2  DH001   2.0  3.0     1.0     NaN  Oxide
3  DH001   3.0  4.0     1.0     4.2  Oxide
4  DH001   4.0  5.0     1.0     4.2  Fresh
5  DH001   5.0  6.0     1.0     NaN  Fresh


In [22]:
test = dhp.drillhole_merge_pipeline(df1, df2)
print(test)

  HOLEID  FROM   TO  LENGTH  Au_ppm   LITH
0  DH001   0.0  1.0     1.0     1.5    NaN
1  DH001   1.0  2.0     1.0     1.5  Oxide
2  DH001   2.0  3.0     1.0     NaN  Oxide
3  DH001   3.0  4.0     1.0     4.2  Oxide
4  DH001   4.0  5.0     1.0     4.2  Fresh
5  DH001   5.0  6.0     1.0     NaN  Fresh
